In [1]:
import os                                   
import pandas as pd
import numpy as np
import torch
import transformers
from scipy.special import expit
from transformers import TrainingArguments,Trainer,DataCollatorWithPadding,AutoModelForSequenceClassification,AutoTokenizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from datasets import Dataset,DatasetDict
from scipy.special import expit
from torch.nn import BCEWithLogitsLoss

Path to dataset files: /kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge


In [2]:
file_path = os.path.join(path, "train.csv")
df = pd.read_csv(file_path)

df.drop('id',axis=1,inplace = True)

df = df.sample(100000).reset_index(drop=True)

In [3]:
raw_ds = Dataset.from_pandas(df)

raw_ds = raw_ds.train_test_split(test_size = 0.2)

train_val = raw_ds['train'].train_test_split(test_size = 0.1)

final_ds = DatasetDict({"train":train_val['train'],"validation":train_val['test'],"test":raw_ds['test']})


In [4]:
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

label_cols = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

def create_labels(example):
    example["labels"] = [float(example[col]) for col in label_cols]
    return example
final_ds = final_ds.map(create_labels)


def tokenize(example):
    return tokenizer(example['comment_text'],truncation = True,max_length=256)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/72000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [5]:
ips = final_ds.map(tokenize,batched = True)

ips = ips.remove_columns(['comment_text'] + label_cols)

ips.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/72000 [00:00<?, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [6]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=6,
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = expit(logits)
    preds = (probs > 0.5).astype(int)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {'f1': macro_f1}

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
labels_np = np.array(ips['train']['labels'])
pos_counts = labels_np.sum(axis=0)
neg_counts = len(labels_np) - pos_counts
raw_weights = neg_counts / (pos_counts + 1e-5)
pos_weight_tensor = torch.tensor(np.log1p(raw_weights), dtype=torch.float32).to(device)

In [11]:
class CustomTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels").float()
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = BCEWithLogitsLoss(pos_weight=self.pos_weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [12]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=ips['train'],
    eval_dataset=ips['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight_tensor
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.094896,0.086330,0.590807
2,0.070495,0.071533,0.653280
3,0.053877,0.072392,0.660761


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3375, training_loss=0.0927310553656684, metrics={'train_runtime': 5432.9352, 'train_samples_per_second': 39.758, 'train_steps_per_second': 0.621, 'total_flos': 2.839339945654733e+16, 'train_loss': 0.0927310553656684, 'epoch': 3.0})

In [13]:
preds = trainer.predict(ips['validation'])
logits = preds.predictions
labels = preds.label_ids
def per_label_threshold(logits,labels):
  if torch.is_tensor(logits):
    logits = logits.cpu().numpy()
  if torch.is_tensor(labels):
      labels = labels.cpu().numpy()
  best_thresh = []
  for i in range(labels.shape[1]):
    probs = 1/(1+np.exp(-(logits[:,i])))
    best_threshold = 0
    best_f1 = 0
    for k in range(100,1000,20):
      thresh = k/1000
      preds = (probs >= thresh).astype(int)
      true = labels[:, i]
      macro_f1 = f1_score(true,preds)
      if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_threshold = thresh
    best_thresh.append(best_threshold)
  return best_thresh
thresholds = per_label_threshold(logits,labels)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [14]:
print("Thresholds per label:", thresholds)

Thresholds per label: [0.74, 0.78, 0.86, 0.82, 0.78, 0.78]


In [15]:
test_preds = trainer.predict(ips['test'])
logits = test_preds.predictions
labels = test_preds.label_ids
probs = 1 / (1 + np.exp(-logits))
thresholds = np.array(thresholds)
final_preds = (probs >= thresholds).astype(int)
baseline_preds = (probs >= 0.5).astype(int)
baseline_f1 = f1_score(labels, baseline_preds, average='macro')
score = f1_score(labels,final_preds,average = 'macro')
print("Baseline F1:", baseline_f1)
print("Calibrated F1:", score)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Baseline F1: 0.6615543315657239
Calibrated F1: 0.6946739481603066


In [16]:
for i, label in enumerate(label_cols):
    base = f1_score(labels[:, i], baseline_preds[:, i])
    new  = f1_score(labels[:, i], final_preds[:, i])
    print(f"{label}: {base:.4f} → {new:.4f}")

toxic: 0.8163 → 0.8352
severe_toxic: 0.4882 → 0.5523
obscene: 0.8173 → 0.8408
threat: 0.5638 → 0.6232
insult: 0.7501 → 0.7774
identity_hate: 0.5336 → 0.5393


In [17]:
from sklearn.metrics import classification_report
print(classification_report(labels, final_preds, target_names=label_cols))

               precision    recall  f1-score   support

        toxic       0.82      0.85      0.84      1956
 severe_toxic       0.48      0.64      0.55       218
      obscene       0.85      0.83      0.84      1084
       threat       0.61      0.64      0.62        67
       insult       0.74      0.82      0.78      1010
identity_hate       0.50      0.58      0.54       178

    micro avg       0.77      0.82      0.79      4513
    macro avg       0.67      0.73      0.69      4513
 weighted avg       0.78      0.82      0.80      4513
  samples avg       0.07      0.08      0.07      4513



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
trainer.save_model("transformer_model_v1")
tokenizer.save_pretrained("transformer_model_v1")
import json
with open("transformer_model_v1/thresholds.json", "w") as f:
    json.dump(thresholds.tolist(), f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
!zip -r transformer_model_v1.zip transformer_model_v1

  adding: transformer_model_v1/ (stored 0%)
  adding: transformer_model_v1/training_args.bin (deflated 53%)
  adding: transformer_model_v1/tokenizer.json (deflated 82%)
  adding: transformer_model_v1/thresholds.json (deflated 39%)
  adding: transformer_model_v1/config.json (deflated 55%)
  adding: transformer_model_v1/tokenizer_config.json (deflated 50%)
  adding: transformer_model_v1/model.safetensors (deflated 7%)
